# Practice Lab: Video & 3D Generation (Lesson 6.6)

Module 6 · 8 exercises · Video + 3D + Safety

Exercises 1, 2, 6, 8 run locally (pure Python).


# Lesson 6.6: Video & 3D Generation

8 exercises: 2E + 3M + 3C

Exercises 1 (tensor math), 2 (RIFE math), 6 (batch catalog), 8 (safety arch) run **locally**.


In [ ]:
import os, csv, json, math


---
## Exercise 1 (Easy): Video Model Comparison

Tensor dimensions + API cost analysis.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
configs = [
    ('5s @ 24fps 1080p', 5, 24, 1080, 1920),
    ('10s @ 24fps 1080p', 10, 24, 1080, 1920),
    ('10s @ 24fps 4K', 10, 24, 2160, 3840),
    ('30s @ 24fps 4K', 30, 24, 2160, 3840),
]

print('Video Tensor Dimensions:')
print(f"{'Config':<25} {'Frames':<8} "
      f"{'Pixels':<15} {'GB (FP32)':<10} "
      f"{'ST Patches'}")
print('=' * 80)
for name, secs, fps, h, w in configs:
    frames = secs * fps
    pixels = frames * h * w * 3
    gb = pixels * 4 / 1e9
    patches = (frames // 4) * (h // 16) * (w // 16)
    print(f'{name:<25} {frames:<8} '
          f'{pixels:>13,} {gb:<10.1f} '
          f'{patches:>10,}')

img_px = 1080 * 1920 * 3
vid_px = 240 * 1080 * 1920 * 3
print(f'\n10s video vs 1 image: {vid_px // img_px}x more pixels')

apis = [
    ('Runway Gen-4.5', 0.48, 5),
    ('Kling 2.6', 0.14, 5),
    ('Veo 3.1', 0.07, 5),
    ('Pika 2.2', 0.32, 5),
    ('Wan 2.2 (self-host)', 0, 5),
]

print('\nAPI Cost (200 x 5s clips):')
print(f"{'Model':<22} {'$/sec':<8} {'$/clip':<10} "
      f"{'200 clips $':<12} {'200 clips INR'}")
print('-' * 68)
for name, cps, dur in apis:
    clip = cps * dur
    batch = clip * 200
    inr = batch * 84
    print(f'{name:<22} ${cps:<7.2f} '
          f'${clip:<9.2f} ${batch:<11.0f} '
          f'Rs {inr:>8,.0f}')

print('\nTraditional (200 vids): Rs 20,00,000')
print('Cheapest API: Veo 3.1 at Rs 5,880')
print('Cheapest overall: Wan 2.2 self-hosted (Rs 0)')


</details>


---
## Exercise 2 (Easy): Frame Interpolation Pipeline

RIFE pass calculations.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import math

src_fps = 8
duration = 5
src_frames = src_fps * duration
targets = [24, 30, 60]

print(f'Source: {src_fps}fps, {duration}s = {src_frames} frames\n')
print(f"{'Target':<10} {'Mult':<8} {'RIFE passes':<14} "
      f"{'Actual fps':<12} {'Frames'}")
print('=' * 58)
for t in targets:
    mult = t / src_fps
    passes = math.ceil(math.log2(mult))
    actual = src_fps * (2 ** passes)
    total = actual * duration
    print(f'{t}fps{"":<5} {mult:.1f}x{"":<4} '
          f'{passes} pass(es){"":<5} '
          f'{actual}fps{"":<6} {total}')

print('\nFile sizes (720p, H.264):')
frame_bytes = 1280 * 720 * 1.5
for t in targets:
    passes = math.ceil(math.log2(t / src_fps))
    actual = src_fps * (2 ** passes)
    total = actual * duration
    raw_mb = total * frame_bytes / 1e6
    h264_mb = raw_mb * 0.02
    print(f'  {t}fps: {total} frames, '
          f'{raw_mb:.0f}MB raw, ~{h264_mb:.1f}MB H.264')

print('\nPipeline order:')
for s in ['1. Generate @ 8-16fps 720p',
          '2. Interpolate (RIFE) -> 30-60fps',
          '3. Upscale (ESRGAN) -> 1080p/4K',
          '4. Color grade + denoise',
          '5. Add audio']:
    print(f'  {s}')


</details>


---
## Exercise 6 (Challenge): Indian Product Video Batch

CSV + metadata + cost analysis.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import csv, json, os

products = [
    {'name': 'Banarasi Silk Saree',
     'hindi': '\u092c\u0928\u093e\u0930\u0938\u0940 \u0938\u093e\u0921\u093c\u0940',
     'category': 'clothing', 'motion': 'orbit', 'duration': 5},
    {'name': 'Gold Jhumka Earrings',
     'hindi': '\u0938\u094b\u0928\u0947 \u0915\u0947 \u091d\u0941\u092e\u0915\u0947',
     'category': 'jewelry', 'motion': 'zoom', 'duration': 5},
    {'name': 'Leather Kolhapuri',
     'hindi': '\u0915\u094b\u0932\u094d\u0939\u093e\u092a\u0941\u0930\u0940 \u091a\u092a\u094d\u092a\u0932',
     'category': 'footwear', 'motion': 'orbit', 'duration': 5},
    {'name': 'Madhubani Painting',
     'hindi': '\u092e\u0927\u0941\u092c\u0928\u0940 \u092a\u0947\u0902\u091f\u093f\u0902\u0917',
     'category': 'art', 'motion': 'dramatic', 'duration': 5},
    {'name': 'Steel Masala Dabba',
     'hindi': '\u092e\u0938\u093e\u0932\u093e \u0921\u092c\u094d\u092c\u093e',
     'category': 'kitchen', 'motion': 'lifestyle', 'duration': 5},
    {'name': 'Brass Ganesh Idol',
     'hindi': '\u092a\u0940\u0924\u0932 \u0917\u0923\u0947\u0936 \u092e\u0942\u0930\u094d\u0924\u093f',
     'category': 'decor', 'motion': 'orbit', 'duration': 5},
    {'name': 'Chanderi Dupatta',
     'hindi': '\u091a\u0902\u0926\u0947\u0930\u0940 \u0926\u0941\u092a\u091f\u094d\u091f\u093e',
     'category': 'clothing', 'motion': 'lifestyle', 'duration': 5},
    {'name': 'Terracotta Diya Set',
     'hindi': '\u091f\u0947\u0930\u093e\u0915\u094b\u091f\u093e \u0926\u0940\u092f\u093e',
     'category': 'decor', 'motion': 'dramatic', 'duration': 5},
    {'name': 'Pashmina Shawl',
     'hindi': '\u092a\u0936\u094d\u092e\u0940\u0928\u093e \u0936\u0949\u0932',
     'category': 'clothing', 'motion': 'zoom', 'duration': 5},
    {'name': 'Dhokra Art Horse',
     'hindi': '\u0922\u094b\u0915\u0930\u093e \u0915\u0932\u093e \u0918\u094b\u0921\u093c\u093e',
     'category': 'art', 'motion': 'orbit', 'duration': 5},
]

os.makedirs('video_batch', exist_ok=True)

with open('video_batch/products.csv', 'w',
          newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=[
        'name', 'hindi', 'category', 'motion', 'duration'])
    w.writeheader()
    w.writerows(products)

print('Product Video Catalog:')
for i, p in enumerate(products):
    print(f'  [{i+1:2d}] {p["name"]:<24} '
          f'({p["hindi"]}) | {p["motion"]}')

for i, prod in enumerate(products):
    meta = {
        'product': prod['name'],
        'hindi': prod['hindi'],
        'category': prod['category'],
        'motion_type': prod['motion'],
        'duration_sec': prod['duration'],
        'api': 'kling-2.6',
        'status': 'pending',
    }
    fname = f"video_batch/{i:02d}_{prod['name'].replace(' ','_')}.json"
    with open(fname, 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

with open('video_batch/products.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
metas = [f for f in os.listdir('video_batch') if f.endswith('.json')]

total_secs = sum(p['duration'] for p in products)
kling_cost = 0.14 * total_secs
kling_inr = kling_cost * 84

print(f'\nCSV verified: {len(rows)} rows')
print(f'Metadata files: {len(metas)}')
print(f'\nBatch cost (Kling): ${kling_cost:.2f} (Rs {kling_inr:,.0f})')
print(f'Traditional: Rs 20,00,000')
print(f'Savings: {(1 - kling_inr/2000000)*100:.0f}%')

# Cleanup
import shutil
shutil.rmtree('video_batch')


</details>


---
## Exercise 8 (Challenge): Safety Compliance System

IT Rules 2026 architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

arch = {
    'regulation': 'IT Amendment Rules 2026',
    'effective': '2026-02-20',
    'layers': [
        {'layer': 1, 'name': 'Pre-Generation Filtering',
         'components': ['Prompt blacklist (NSFW/CSAM)',
                        'Celebrity name detection',
                        'Semantic intent analysis'],
         'latency': '<100ms'},
        {'layer': 2, 'name': 'Generation-Time Watermarking',
         'components': ['C2PA v2.3 metadata (ISO standard)',
                        'SynthID pixel watermark (99.3%)',
                        'Video Seal temporal watermark'],
         'latency': '<500ms'},
        {'layer': 3, 'name': 'Post-Generation Moderation',
         'components': ['NSFW classifier (CLIP-based)',
                        'Face detection (RetinaFace)',
                        'Celebrity match (ArcFace)',
                        'Violence/gore detection',
                        'AI-Generated label overlay'],
         'latency': '<2s/video'},
        {'layer': 4, 'name': 'Takedown System',
         'components': ['2-hour: intimate AI imagery',
                        '3-hour: other unlawful content',
                        'Automated flagging queue',
                        'Human review escalation',
                        'Re-upload hash matching'],
         'sla': '99.9% within deadline'},
        {'layer': 5, 'name': 'Compliance Reporting',
         'components': ['Monthly SGI transparency report',
                        'Quarterly user notifications',
                        'Annual audit trail',
                        '90-day data retention'],
         'retention': '90 days'},
    ],
    'costs': {
        'infra': '15-20% of total platform cost',
        'moderation': 'Rs 2-5L/month',
        'watermark': 'Rs 0.50/video',
        'detection': 'Rs 1-2/video',
        'annual_total': 'Rs 40-75L',
    },
    'penalties': {
        'safe_harbor_loss': 'Section 79 removed',
        'max_fine': 'Rs 250 Cr (~$30M) under DPDPA',
    },
}

print('AI Video Safety Architecture (IT Rules 2026)')
print('=' * 50)
print(f"Regulation: {arch['regulation']}")
print(f"Effective: {arch['effective']}")

for layer in arch['layers']:
    print(f"\nLayer {layer['layer']}: {layer['name']}")
    for c in layer['components']:
        print(f'  - {c}')
    for k in ['latency', 'sla', 'retention']:
        if k in layer:
            print(f'  [{k}: {layer[k]}]')

print('\nCosts:')
for k, v in arch['costs'].items():
    print(f'  {k}: {v}')

print('\nPenalties:')
for k, v in arch['penalties'].items():
    print(f'  {k}: {v}')

total = sum(len(l['components']) for l in arch['layers'])
print(f'\nLayers: {len(arch["layers"])}')
print(f'Total components: {total}')


</details>


---
## Exercise 3 (Medium): Image-to-3D Comparison
Needs API/tools. See practice-lab-6_6.html.


In [ ]:
# Exercise 3: Image-to-3D Comparison
pass


---
## Exercise 4 (Medium): Video Post-Processing Pipeline
Needs API/tools. See practice-lab-6_6.html.


In [ ]:
# Exercise 4: Video Post-Processing Pipeline
pass


---
## Exercise 5 (Medium): 3D Asset for Game Engine
Needs API/tools. See practice-lab-6_6.html.


In [ ]:
# Exercise 5: 3D Asset for Game Engine
pass


---
## Exercise 7 (Challenge): Gaussian Splatting Capture
Needs API/tools. See practice-lab-6_6.html.


In [ ]:
# Exercise 7: Gaussian Splatting Capture
pass
